# System to Attention
由GPT-4產生
從gemini 評估的prompt 表現較差的組別
"全面評估一項業務創新計畫的可行性，並探討其對目標市場的影響。"
"仔細驗證新業務計畫的各個部分，並預測其對公司股價的可能影響。"
"深入探討業務創新計畫如何影響投資者的預期，並全面評估其長期效益。"

從gemini 評估的prompt 表現較好的組別
"評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險。"
"分析新業務計畫對公司股價的可能影響，並審慎評量其對投資者的吸引力。"
"探討業務創新計畫如何影響投資者的預期回報，並審慎評量其對公司財務狀況的影響。"

In [2]:
# %pip install pandas
# %pip install numpy
# %pip install langchain_google_genai

In [3]:
import os
import textwrap
import math
import pandas as pd
import numpy as np

from datetime import datetime
from dateutil.relativedelta import relativedelta

import mplfinance as mpf
import google.generativeai as genai
from langchain.schema.messages import HumanMessage
from langchain.schema.messages import SystemMessage
from langchain.schema.messages import AIMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory

from IPython.display import display
from IPython.display import Markdown

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                convert_system_message_to_human=True,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

/Users/yanyifu/Documents/_Coding/LLM Predict Stock/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
def llm_response(llm, text):
    ''' 
        filepath: string 檔案路徑
        text: string 問題指令
        
        return: string 回答
    '''

    try:
        response = llm.invoke(text)
        return response.content
    except Exception as e:
        error = "error: " + str(e)
        print(error)
        pass
    
    return error

In [5]:
# 轉換回答變成訊號
def response_to_signal(text):
    '''
        text: string 回答訊息
        
        return: int 回答訊號
    '''
    if text.find('no') > 0:
        signal = -1
    elif text.find('yes') > 0:
        signal = 1
    else:
        signal = 0

    return signal

# 設定要測試的股票

In [6]:
# batch = []

# def response1(llm, paths):
#     for batch_idx in range(len(paths)):


#     for res in res_s2a:
#         with open('response.txt', 'w') as f:
#             f.write(res.content)
#     return res_s2a

# def save_file()

# # 將 AI 的回應添加到消息列表中
# messages.append(res_s2a)
# messages.append(
#     HumanMessage(
#         content=[
#             {
#                 "type": "text",
#                 "text": "請用上述資訊對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown",
#             },
#         ]
#     )
# )


# res = llm.batch([messages])
# print(res.content)

In [7]:
import time
import os

def run(stock_id, st, et):
    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    print(folder_path)
    sorted_news_files = sorted(os.listdir(folder_path))
    sorted_news_files = [file for file in sorted_news_files if st <= datetime.strptime(file, '%Y%m%d') <= et]
    file_id = 0
    batch_size = 15

    while file_id < len(sorted_news_files):
        batch = []
        for i in range(batch_size):

            if file_id + i >= len(sorted_news_files):
                break
            with open(folder_path + sorted_news_files[file_id+i], 'r', encoding='utf-8') as f:
                news = f.read()

            # 設置初始消息列表，包括 SystemMessage 和第一個 HumanMessage
            messages = [
                SystemMessage(content="你是一個厲害的股市操盤手，專長是當日沖消"),
                HumanMessage(content="評估一項業務創新計畫如何改變目標市場的格局，並審慎評量其實施的風險，並保留相關的資訊。" + news)
            ]
            batch.append(messages)
            print(file_id+i)    

        print(st, et)
        res_s2a = llm.batch(batch)

        out_path = os.path.abspath(os.getcwd()) + "out_SBP/" + stock_id + "/"
        os.makedirs(out_path, exist_ok=True)
        for i in range(len(res_s2a)):
            with open(out_path + sorted_news_files[file_id+i] + ".txt" , 'w') as f:
                f.write(res_s2a[i].content)
        time.sleep(1)
        file_id += batch_size

In [8]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
from datetime import datetime

start_day = datetime(2023, 1, 1)
end_day = datetime(2023, 12, 31)

stock_ids = ['2880', '2211', '2707']
stock_names = ['華南金', '長榮鋼', '晶華']

# stock_ids = ['2211']
# stock_names = ['長榮鋼']


# stock_ids = ['2618', '2881', '2880', '2211', '2707']
# stock_names = ['長榮航', '富邦金', '華南金', '長榮鋼', '晶華']

# stock_ids = ['9941', '2308', '1514', '1101', '2542']
# stock_names = ['裕融', '台達電', '亞力', '台泥', '興富發']
print(123)
for idx in range(len(stock_ids)):
    print(f"run stock {stock_ids[idx]} done")
    run(stock_ids[idx], start_day, end_day)

In [ ]:
count = 0
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id}')
    # qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()

                res_s2a = llm_response(llm, f"""<s> [INST] 請在以下新聞中，提取偏見的部分，並保留他們的意見，回傳該文章中含有偏見的內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)
                
                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                
                sig = response_to_signal(res)   

                count += 2
                break
            
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    # ========================================================================================
    folder_path = "eval_s2a/output0603_morebias/"
    # ========================================================================================
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")

print(count)

# 對照組

In [ ]:
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id}')
    # qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue

        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()
                
                res_s2a = llm_response(llm, f"""<s> [INST] 給定以下新聞，提取無偏見的部分，而不是他們的意見，這樣使用該文章將是為新聞的問題部分提供無偏見答案的良好內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)

                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                 
                sig = response_to_signal(res) 
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    # ========================================================================================
    folder_path = "eval_s2a/output0605/"
    # ========================================================================================
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")